# Ch 32. 파인튜닝이 필요한 경우와 아닌 경우

**[원본 웹페이지](https://desty.github.io/study-ai-assistant-engineering/part7/32-when-to-finetune/)**

## Abstract

- **"AI 프로젝트 = 파인튜닝" 가정이 왜 틀린가**
- 4 게이트 **결정 트리** — 게이트 하나만 막혀도 보류
- **Prompt · RAG · SFT · DPO** 4 접근의 장단·비용·시간 비교
- 파인튜닝이 잘 푸는 문제 vs 못 푸는 문제
- **데이터 요구량** — 수백 / 수천 / 수만 의 의미
- 6대 함정 (RAG 시도 안 함 · 데이터 비용 과소 · 한 번 학습으로 끝 · 평가 부재 · 새 사실을 SFT 로 · base 모델로 SFT 시작)

## Dependencies

In [ ]:
!pip install -q anthropic openai chromadb langchain langgraph langchain-anthropic langchain-openai

## API Keys

In [ ]:
import os
from getpass import getpass

for k in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY']:
    if not os.getenv(k):
        os.environ[k] = getpass(f'{k}: ')

---

## 1. 개념 — 가장 중요한 질문은 "파인튜닝 안 해도 되는가"

대부분의 LLM 프로젝트에서 **파인튜닝은 마지막 카드**입니다. 처음부터 파인튜닝을 가정하고 시작하면:

- 데이터 라벨링에 수개월 소요
- GPU 비용에 수백~수천만 원
- 학습 끝나도 base 모델 업그레이드 시 다시 학습
- 평가셋 없으면 회귀를 모름

이 모든 비용이 **Prompt + RAG 만으로 풀렸을 문제** 에 들어갈 수 있습니다.


| 게이트 | 통과 기준 | 막히면 |
|---|---|---|
| ① Prompt + RAG 로 풀리나? | "안 풀린다" 가 평가셋으로 입증 | Prompt/RAG 만 |
| ② 평가셋 충분한가? | 수K 이상의 (q, gold answer) | 평가셋부터 (Ch 16) |
| ③ 데이터 수집·라벨 비용 < 절감? | 1년 ROI 계산 양수 | 비용 모델 재검토 |
| ④ 매 분기 재학습 운영 가능? | 파이프라인 + GPU 예산 + 책임자 | API + prompt 로 |

**4개 게이트를 모두 통과해야** 파인튜닝 검토 시작입니다. 통과 후에도 PoC (작은 데이터로 LoRA) 로 효과를 측정한 뒤 본 학습으로 갑니다.

---

## 2. 왜 필요한가 — Prompt/RAG 의 천장 두 개

다음 두 천장이 보이면 SFT 검토가 정당화됩니다.

**① 톤·말투·출력 형식이 안정 안 됨**. Prompt + few-shot 으로 80% 까지 가지만 5% 의 outlier 가 항상 다른 형식. Production 에선 5% 도 큰 비용. SFT 가 가장 잘 푸는 영역.

**② 모델 크기 비용이 한계**. Opus 로 풀리는데 Haiku 로는 안 풀림 → Haiku 를 SFT 해서 풀리게 만들면 **장기 비용 절감** 이 SFT 비용을 능가. Ch 30 의 모델 라우팅을 더 공격적으로.

반대로 **새 사실 주입** 은 SFT 의 영역이 아닙니다 — RAG 가 압도적으로 효율적. SFT 로 사실을 넣으려면 다시 학습 필요, 사실 갱신마다 큰 비용.

> "Fine-tuning teaches **how**, RAG provides **what**." — 업계 통념

---

## 3. 어디에 쓰이는가 — 4 접근법 비교


| 접근 | 좋음 | 약함 | 비용 | 시간 |
|---|---|---|---|---|
| **① Prompt** | 형식 변경 · 단일 작업 | 도메인 사실 · 대규모 톤 변환 | $ | 분 |
| **② RAG** | 도메인 사실 · 최신 · citation | 톤 · 형식 강제 | $$ | 일 |
| **③ SFT (LoRA)** | 톤 · 출력 형식 · 분류 | 새 사실 주입 | $$$ | 주 |
| **④ DPO/RLHF** | 안전 · 정중함 · 거절 정책 | 데이터 수집 · 과적합 | $$$$ | 월 |

**시도 순서**: ①→②→③→④. 각 단계는 **앞 단계가 천장에 부딪힌 게 평가셋으로 증명** 됐을 때만 다음으로.

### 사례별 첫 시도

| 문제 | 첫 시도 | 다음 |
|---|---|---|
| "회사 톤으로 답해주세요" | ① Prompt + few-shot | 안 되면 ③ SFT |
| "사내 위키 기반 답변" | ② RAG | citation 정확도 천장 시 ③ |
| "법률 문서 5분류" | ① Prompt → ③ SFT (Haiku) | 비용 절감용 |
| "새 정책에 따라 거절 톤 변경" | ④ DPO (소량) | SFT 로는 안정 안 될 때 |
| "실시간 주가 답변" | ② RAG (도구 호출) | 절대 ③ 아님 |
| "JSON 출력 형식 안정" | ① + tool-use (Ch 6) | 안 되면 ③ |

---

## 4. 최소 예제 — 파인튜닝 필요성 체크리스트 10문항

In [ ]:
QUESTIONS = [
    ("평가셋이 1000개 이상 있는가?", "no"),                          # (1)!
    ("Prompt + few-shot 5개로 70% 이상 통과 못 하는가?", "no"),
    ("RAG 로 풀어봤는가?", "no"),                                    # (2)!
    ("실패 케이스가 '톤·형식·분류' 중 하나에 집중되는가?", "no"),
    ("실패가 '사실 정확도' 가 주된 원인인가?", "yes"),                # (3)!
    ("학습 데이터 (q,a) 1K+ 만들 라벨러·예산 확보됐는가?", "no"),
    ("기존 모델보다 작은 모델로 운영해 비용 절감하려는가?", "no"),    # (4)!
    ("정렬·거절 톤 같은 안전 정책 변경이 필요한가?", "no"),
    ("매 분기 재학습 + 평가 파이프라인 책임자 있는가?", "no"),
    ("LoRA PoC (100개 샘플) 로 효과 본 적 있는가?", "no"),
]

# yes 가 7개 이상이면 SFT 검토 시작
yes_count = sum(1 for _, a in QUESTIONS if a == "yes")
print(f"YES: {yes_count}/10 → {'검토 시작' if yes_count >= 7 else '아직 일러요'}")

1. 평가셋 없으면 학습 후 효과 측정 불가 → 무조건 Ch 16 먼저.
2. RAG 시도하지 않은 채 SFT 가는 건 90% 헛수고.
3. **사실 정확도가 주 원인이면 RAG 가 답** — SFT 가 아니라.
4. 비용 절감 목적의 SFT 는 정당한 이유 (Haiku 를 도메인에 맞춤).

YES 가 7개 미만이면 그 항목들을 먼저 채워라. 6개에서 SFT 시작하면 사고 확률이 큽니다.

---

## 5. 실전 — 데이터 요구량 감각

| 작업 유형 | 최소 | 권장 | 좋은 결과 |
|---|---:|---:|---:|
| 분류 (10 클래스) | 500 | 2K | 5K |
| 톤 변환 (style transfer) | 1K | 5K | 20K |
| 도메인 SFT (chat) | 5K | 20K | 100K |
| Instruction tuning (general) | 50K | 200K | 1M+ |
| DPO (선호) | 1K | 5K | 20K |

**"모르면 5K 부터"** 가 안전한 시작점. 이보다 적으면 효과 측정 노이즈가 큼.

### 데이터 품질이 양보다 중요

| 함정 | 결과 |
|---|---|
| 한 라벨러가 다 라벨 | 그 사람 편향이 모델에 박힘 |
| 자동 생성만 | 모델 자체 편향 증폭 |
| 노이즈 5% | 학습 안 됨 (소량 학습일수록 치명) |
| Eval set 누출 | 학습 점수만 좋고 실전은 나쁨 |

라벨링 가이드라인 (Ch 16) 을 SFT 데이터에도 같은 수준으로 적용.

### PoC 단계 — 100~500 개로 효과 측정

본 학습 가기 전 **PoC**:

1. 100~500개 작은 학습셋 (실데이터)
2. LoRA r=8 (가벼움 — Ch 33)
3. 평가셋에서 baseline (no FT) 대비 +5pt 이상 → 진짜 학습 단계로
4. +5pt 미만 → 데이터 늘려도 한계 — 접근법 재검토

**+5pt** 는 도메인마다 조정. 핵심은 "PoC 단계에서 신호가 보이는가" 의 yes/no.

### 비용 모델 — 1년 ROI

In [ ]:
SFT 비용 = 라벨링비 + GPU 학습비 + 평가/회귀 운영비
        ≈ (5K × $1)  + ($500)   + ($200/월 × 12)  ≈ $7900

연간 절감 = (Opus 비용 - Haiku 비용) × 연간 호출 수
        예: ($0.030 - $0.001) × 1M 호출 = $29,000

ROI = 29000 / 7900 ≈ 3.7  → 양수. 단, 품질이 동등해야 함.

ROI 가 1.0 미만이면 **하지 말 것**. 운영 부담까지 고려하면 1.5 이상은 되어야 안전.

---

## 6. 자주 깨지는 포인트

- **RAG 안 해보고 SFT 로 직행**. 사실 정확도 문제는 RAG 가 잘 풀음. SFT 로 사실 주입은 비싸고 갱신 어려움.
- **데이터 비용 과소 평가**. 5K (q,a) 만들기는 의외로 큰 작업 — 라벨러 1명이 하루 50개면 100일. 외부 라벨링 회사 비용 ≈ 건당 $1~5.
- **한 번 학습으로 끝**. 베이스 모델이 분기마다 업그레이드 → 우리 SFT 도 다시. 인프라 + 책임자 없이는 stale 됨.
- **평가셋 부재**. 학습 후 효과 측정 불가 → 회귀 발생해도 모름. **평가셋이 학습 데이터보다 먼저**.
- **새 사실을 SFT 로 주입**. 한 번 들어간 사실은 갱신이 안 됨 → 다음 변경마다 재학습. RAG 로 분리.
- **Base 모델로 SFT 시작**. Instruct 모델이 이미 SFT 한 번 끝낸 상태 — 우리는 그 위에 도메인 SFT 만. base 부터 시작하면 데이터 100배 필요.
- **PoC 없이 본 학습**. 5K 데이터 만들고 학습했는데 효과 0pt → 큰 손실. 100개 PoC 가 항상 먼저.
- **품질을 토큰 단위 metric (loss · perplexity) 로만 본다**. 우리 도메인 평가셋의 정확도/F1 이 정답. loss 는 보조.

---

## 7. 운영 체크리스트

- [ ] 4 게이트 결정 트리 통과 여부 문서화
- [ ] 평가셋 1K+ 가 학습 시작 전 준비
- [ ] PoC (100~500 샘플) 효과 측정 후 본 학습
- [ ] 비용 모델 (1년 ROI) 양수 확인
- [ ] 학습 데이터에 평가셋 누출 없음 검사
- [ ] 라벨러 N명 + 가이드라인 + 일관성 체크
- [ ] 분기별 재학습 + 평가 파이프라인 + 책임자 지정
- [ ] Base vs Instruct 모델 선택 명확
- [ ] 학습 후 도메인 평가셋 + safety regression 두 축 측정
- [ ] 회귀 발생 시 즉시 롤백 가능한 모델 버전 관리

---

## 8. 연습문제 & 다음 챕터

1. 사내 IT 헬프데스크 챗봇이 "내부 정책 답변 정확도 60%" 라는 문제가 있다. 4 게이트를 적용해 첫 시도가 무엇이어야 하는지 결정하라.
2. "회사 공식 톤" 으로 답을 바꾸고 싶다. Prompt few-shot · RAG · SFT 셋 중 어느 것부터, 왜?
3. SFT 비용 = $7900, 연간 호출 = 100K (1M 의 1/10) 인 경우 ROI 를 계산하고 결정하라.
4. PoC (100 샘플) 결과 baseline +1pt 만 나왔다. 다음 액션 3가지 옵션을 적어라.

**다음 챕터** — 결정 트리를 통과했다면, 실제로 손에 묻혀 본다. [Ch 33 LoRA / QLoRA 실전](33-lora-qlora.md) 으로.

---

## 원전

- Stanford CME 295 — *Transformers and LLMs* Lec 4
- Hu et al. (2021) *LoRA: Low-Rank Adaptation of Large Language Models*
- OpenAI · Anthropic — fine-tuning best practice docs
- 업계 가이드: 데이터 수집 · 라벨링 · ROI 계산